In [6]:
import json
import gurobipy as gp
from gurobipy import GRB, quicksum

### 학번: 2025324096
### 이름: 장서현
### Variable 선정 이유: Assignment and Positional Date Variables
#### Sequence-dependent setup time $s_{ij}$은 작업 $i$ 바로 뒤에 $j$가 오는 경우에만 발생하므로, “즉시 인접” 관계를 명확히 표현할 수 있어야 한다. Assignment/Position 변수는 각 작업의 순서를 위치로 직접 배정하여, 인접한 두 위치 사이에서 setup time을 반영할 수 있다고 판단했다.
### Formulation은 IDO6001_HW4.pdf (Problem 1.2)에 첨부하였습니다. 

## Dataset
### Set
- N = $\{1,2,...,n\}, j \in N$
### Parameters
- $p_j$: processing time of job $j$
- $d_j$: due date of job $j$
- $w_j$: weight of job $j$
- $s_{i,j}$: setup time if job $i$ precedes job $j$

In [7]:
with open('single_setup_dataset.json') as file:
    data = json.load(file)

print(len(data['N']))
print(data['p'])
print(data['d'])
print(data['w'])
print(data['s']) 

8
{'0': 88, '1': 91, '2': 65, '3': 22, '4': 63, '5': 13, '6': 9, '7': 70}
{'0': 181, '1': 252, '2': 350, '3': 174, '4': 307, '5': 215, '6': 94, '7': 180}
{'0': 3, '1': 2, '2': 5, '3': 2, '4': 7, '5': 1, '6': 5, '7': 2}
{'0': [0, 26, 25, 4, 14, 20, 25, 22], '1': [25, 0, 35, 23, 33, 8, 42, 12], '2': [22, 33, 0, 18, 12, 26, 13, 19], '3': [4, 24, 21, 0, 10, 24, 22, 18], '4': [15, 28, 12, 11, 0, 29, 13, 22], '5': [26, 8, 32, 22, 32, 0, 39, 9], '6': [29, 41, 12, 25, 14, 38, 0, 31], '7': [20, 14, 23, 19, 25, 7, 30, 0]}


In [8]:
n = len(data['N']) # 작업의 개수

# Sets
N = [_ for _ in range(n)] # 작업 set을 리스트 자료구조를 이용해 0, 1, 2,..., n-1 까지 담아줌

# Parameters
p = [data['p'][f'{i}'] for i in N] # 작업의 가공시간 데이터를 리스트 자료구조를 이용해 만들어 줌
d = [data['d'][f'{i}'] for i in N] # 작업의 마감 기한 데이터를 리스트 자료구조를 이용해 만들어 줌
w = [data['w'][f'{i}'] for i in N] # 작업의 가중치 데이터를 리스트 자료구조를 이용해 만들어 줌
s = [data['s'][f'{i}'] for i in N] # 작업의 setup time 데이터를 리스트 자료구조를 이용해 만들어 줌 ex) 작업 1 -> 2 에는 setup time 26 필요

M = (sum(p) + sum(max(s[i]) for i in N)) * 2 # big M
print(p)
print(d)
print(w)
print(s)

[88, 91, 65, 22, 63, 13, 9, 70]
[181, 252, 350, 174, 307, 215, 94, 180]
[3, 2, 5, 2, 7, 1, 5, 2]
[[0, 26, 25, 4, 14, 20, 25, 22], [25, 0, 35, 23, 33, 8, 42, 12], [22, 33, 0, 18, 12, 26, 13, 19], [4, 24, 21, 0, 10, 24, 22, 18], [15, 28, 12, 11, 0, 29, 13, 22], [26, 8, 32, 22, 32, 0, 39, 9], [29, 41, 12, 25, 14, 38, 0, 31], [20, 14, 23, 19, 25, 7, 30, 0]]


## Code

In [9]:
# Positions 정의
K = range(n)

In [10]:
model = gp.Model("Problem 1.2")
model.setParam('OutputFlag', 0)

# Decision Variables 정의
u = model.addVars(N, K, vtype=GRB.BINARY, name="u")
gamma = model.addVars(K, vtype=GRB.CONTINUOUS, lb=0, name="gamma")
T = model.addVars(N, vtype=GRB.CONTINUOUS, lb=0, name="T")
z = model.addVars(N, N, range(1, n), vtype=GRB.BINARY, name="z")

# Objective Function 정의
model.setObjective(quicksum(w[j] * T[j] for j in N), GRB.MINIMIZE)

# Constraints 정의
for j in N: model.addConstr(quicksum(u[j, k] for k in K) == 1, name=f"AssignJob_{j}")
for k in K: model.addConstr(quicksum(u[j, k] for j in N) == 1, name=f"AssignPos_{k}")
for k in range(1, n):
    setup_time_term = quicksum(s[i][j] * z[i, j, k] for i in N for j in N if i != j)
    processing_time_term = quicksum(p[j] * u[j, k] for j in N)
    model.addConstr(gamma[k] >= gamma[k-1] + processing_time_term + setup_time_term, name=f"CompletionTime_Pos_{k}")
for k in range(1, n):
    for i in N:
        for j in N:
            if i == j: continue
            model.addConstr(z[i, j, k] >= u[i, k-1] + u[j, k] - 1, name=f"Lin_Lower_{i}_{j}_{k}")
            model.addConstr(z[i, j, k] <= u[i, k-1], name=f"Lin_Upper1_{i}_{j}_{k}")
            model.addConstr(z[i, j, k] <= u[j, k], name=f"Lin_Upper2_{i}_{j}_{k}")
model.addConstr(gamma[0] >= quicksum(p[j] * u[j, 0] for j in N), name="CompletionTime_Pos_0") # 초기 셋업(s_0j)은 데이터에 없으므로 0으로 가정
for j in N:
    for k in K:
        model.addConstr(T[j] >= gamma[k] - d[j] - M * (1 - u[j, k]), name=f"Tardiness_{j}_Pos_{k}")

# 최적화 실행
model.optimize()

In [11]:
# 결과 출력
if model.status == GRB.OPTIMAL:
    print(f"Optimal Objective Value: {model.objVal}")
    
    sequence = [None] * n
    for k in K:
        for j in N:
            if u[j, k].X > 0.5:
                sequence[k] = j
                break
    
    print(f"Optimal Sequence: {sequence}")
    
    print("-" * 60)
    print(f"{'Pos':<5} | {'Job':<5} | {'Proc':<5} | {'Setup':<5} | {'Compl':<8} | {'Due':<5} | {'Tard':<5}")
    print("-" * 60)
    
    prev_job = None
    
    for k in K:
        job = sequence[k]
        comp_time = gamma[k].X
        tardiness = T[job].X
        
        setup = 0
        if k > 0:
            setup = s[prev_job][job]
        
        print(f"{k:<5} | {job:<5} | {p[job]:<5} | {setup:<5} | {comp_time:<8.1f} | {d[job]:<5} | {tardiness:<5.1f}")
        prev_job = job

else:
    print("Optimal solution not found.")

Optimal Objective Value: 1129.0
Optimal Sequence: [6, 3, 0, 5, 4, 2, 7, 1]
------------------------------------------------------------
Pos   | Job   | Proc  | Setup | Compl    | Due   | Tard 
------------------------------------------------------------
0     | 6     | 9     | 0     | 9.0      | 94    | -0.0 
1     | 3     | 22    | 25    | 56.0     | 174   | -0.0 
2     | 0     | 88    | 4     | 148.0    | 181   | -0.0 
3     | 5     | 13    | 20    | 181.0    | 215   | -0.0 
4     | 4     | 63    | 32    | 276.0    | 307   | -0.0 
5     | 2     | 65    | 12    | 353.0    | 350   | 3.0  
6     | 7     | 70    | 19    | 442.0    | 180   | 262.0
7     | 1     | 91    | 14    | 547.0    | 252   | 295.0
